Import Libraries

In [21]:
import pandas as pd
import numpy as np
from itertools import combinations
from IPython.display import display

In [22]:
import os
import glob

print("Current Working Directory:")
print(os.getcwd())

print("\nCSV files available:")
print(glob.glob("*.csv"))

print("\nAll files:")
print(os.listdir())

Current Working Directory:
c:\Users\JAKKANA HASINI\OneDrive\Desktop\MACHINE LEARNING

CSV files available:
['tennis_gini_all_splits.csv', 'tennis_gini_best_split_by_feature.csv', 'unique_tennis_gini_dataset.csv']

All files:
['hasini_jakkana.ipynb', 'tennis_gini_all_splits.csv', 'tennis_gini_best_split_by_feature.csv', 'unique_tennis_gini_dataset.csv']


Load Dataset Automatically

In [23]:
import pandas as pd
import glob

csv_files = glob.glob("*.csv")

print("Available CSV Files:", csv_files)

if len(csv_files) > 0:
    df = pd.read_csv(csv_files[0])
    print("\nDataset Loaded Successfully!")
    display(df)
else:
    print("No CSV file found. Check your folder location.")

Available CSV Files: ['tennis_gini_all_splits.csv', 'tennis_gini_best_split_by_feature.csv', 'unique_tennis_gini_dataset.csv']

Dataset Loaded Successfully!


,Feature,Left Categories,Right Categories,Parent Gini,Left Child Gini,Right Child Gini,Weighted Child Gini,Gini Reduction
0,Outlook,"('Rainy',)",['Sunny'],0.46875,0.50000,0.37500,0.43750,0.03125
1,Court,"('Dry',)",['Wet'],0.46875,0.50000,0.37500,0.43750,0.03125
2,Wind,"('Calm',)",['Windy'],0.46875,0.37500,0.00000,0.18750,0.28125
3,Temperature,"('Cool',)",['Warm'],0.46875,0.46875,0.46875,0.46875,0.00000


Identify Features and Target

In [24]:
target_column = "Play"

features = [
    col for col in df.columns
    if col != target_column
]

print("Target Column:", target_column)

print("\nInput Features:")
print(features)

print("\nTarget Class Distribution:")
print(df[target_column].value_counts())

Target Column: Play

Input Features:
['Feature', 'Left Categories', 'Right Categories', 'Parent Gini', 'Left Child Gini', 'Right Child Gini', 'Weighted Child Gini', 'Gini Reduction']

Target Class Distribution:


KeyError: 'Play'

Define Gini Impurity Function

In [ ]:
def calculate_gini(data):

    probabilities = data.value_counts(normalize=True)

    gini = 1 - sum(probabilities ** 2)

    return gini

Calculate Parent Gini Impurity

In [ ]:
parent_gini = calculate_gini(df[target_column])

print("Total Records:", len(df))

print("\nParent Gini Impurity:",
      round(parent_gini, 4))

Total Records: 16

Parent Gini Impurity: 0.4688


Generate All Possible Splits

In [ ]:
from itertools import combinations

results = []

for feature in features:

    categories = sorted(df[feature].dropna().unique())

    for r in range(1, len(categories)):

        for subset in combinations(categories[1:], r - 1):

            left_categories = (categories[0],) + subset

            right_categories = [
                value for value in categories
                if value not in left_categories
            ]

            left = df[df[feature].isin(left_categories)]

            right = df[~df[feature].isin(left_categories)]

            if len(left) == 0 or len(right) == 0:
                continue

            results.append({
                "Feature": feature,
                "Left Categories": left_categories,
                "Right Categories": right_categories,
                "Left Data": left,
                "Right Data": right
            })

print("Total Possible Splits:", len(results))

Total Possible Splits: 4


Calculate Left Child Gini Impurity

In [ ]:
for split in results:

    left_gini = calculate_gini(
        split["Left Data"][target_column]
    )

    split["Left Gini"] = left_gini

print("Left Child Gini Impurity Calculated Successfully!")

Left Child Gini Impurity Calculated Successfully!


Calculate Right Child Gini Impurity

In [ ]:
for split in results:

    right_gini = calculate_gini(
        split["Right Data"][target_column]
    )

    split["Right Gini"] = right_gini

print("Right Child Gini Impurity Calculated Successfully!")

Right Child Gini Impurity Calculated Successfully!


Calculate Weighted Child Gini Impurity

In [ ]:
total_records = len(df)

for split in results:

    left_weight = len(split["Left Data"]) / total_records

    right_weight = len(split["Right Data"]) / total_records

    weighted_gini = (
        left_weight * split["Left Gini"] +
        right_weight * split["Right Gini"]
    )

    split["Weighted Child Gini"] = weighted_gini

print("Weighted Child Gini Calculated Successfully!")

Weighted Child Gini Calculated Successfully!


Calculate Gini Reduction

Formula:
\[
Gini\ Reduction =
Gini(parent)-Gini(weighted)
\]

In [ ]:
for split in results:

    gini_reduction = (
        parent_gini - split["Weighted Child Gini"]
    )

    split["Parent Gini"] = parent_gini

    split["Gini Reduction"] = gini_reduction

print("Gini Reduction Calculated Successfully!")

Gini Reduction Calculated Successfully!


Display Gini Calculations for All Splits

In [ ]:
results_df = pd.DataFrame([
    {
        "Feature": s["Feature"],
        "Left Categories": s["Left Categories"],
        "Right Categories": s["Right Categories"],
        "Parent Gini": s["Parent Gini"],
        "Left Child Gini": s["Left Gini"],
        "Right Child Gini": s["Right Gini"],
        "Weighted Child Gini": s["Weighted Child Gini"],
        "Gini Reduction": s["Gini Reduction"]
    }
    for s in results
])

print("GINI CALCULATION FOR ALL SPLITS")

display(results_df.round(4))

GINI CALCULATION FOR ALL SPLITS


,Feature,Left Categories,Right Categories,Parent Gini,Left Child Gini,Right Child Gini,Weighted Child Gini,Gini Reduction
0,Outlook,"(Rainy,)",[Sunny],0.4688,0.5000,0.3750,0.4375,0.0312
1,Court,"(Dry,)",[Wet],0.4688,0.5000,0.3750,0.4375,0.0312
2,Wind,"(Calm,)",[Windy],0.4688,0.3750,0.0000,0.1875,0.2812
3,Temperature,"(Cool,)",[Warm],0.4688,0.4688,0.4688,0.4688,0.0000


Find the Best Split
The split with the highest Gini reduction is considered the best split.

In [ ]:
best_split = results_df.loc[
    results_df["Gini Reduction"].idxmax()
]

print("BEST SPLIT")

display(best_split.to_frame())

print("\nBest Feature:", best_split["Feature"])

print("Best Gini Reduction:",
      round(best_split["Gini Reduction"], 4))

BEST SPLIT


,2
Feature,Wind
Left Categories,"(Calm,)"
Right Categories,[Windy]
Parent Gini,0.46875
Left Child Gini,0.375
Right Child Gini,0.0
Weighted Child Gini,0.1875
Gini Reduction,0.28125



Best Feature: Wind
Best Gini Reduction: 0.2812


Find the Best Split for Each Feature

In [ ]:
best_by_feature = results_df.loc[
    results_df.groupby("Feature")["Gini Reduction"].idxmax()
]

print("BEST SPLIT FOR EACH FEATURE")

display(best_by_feature.round(4))

BEST SPLIT FOR EACH FEATURE


,Feature,Left Categories,Right Categories,Parent Gini,Left Child Gini,Right Child Gini,Weighted Child Gini,Gini Reduction
1,Court,"(Dry,)",[Wet],0.4688,0.5000,0.3750,0.4375,0.0312
0,Outlook,"(Rainy,)",[Sunny],0.4688,0.5000,0.3750,0.4375,0.0312
3,Temperature,"(Cool,)",[Warm],0.4688,0.4688,0.4688,0.4688,0.0000
2,Wind,"(Calm,)",[Windy],0.4688,0.3750,0.0000,0.1875,0.2812


Save Results into CSV Files

In [ ]:
results_df.to_csv("tennis_gini_all_splits.csv", index=False)

best_by_feature.to_csv(
    "tennis_gini_best_split_by_feature.csv",
    index=False
)

print("Both CSV files saved successfully!")

Both CSV files saved successfully!


Final Conclusion

In [ ]:
print("ASSIGNMENT CONCLUSION")

print("Parent Gini Impurity:",
      round(parent_gini, 4))

print("Best Feature:",
      best_split["Feature"])

print("Best Gini Reduction:",
      round(best_split["Gini Reduction"], 4))

print("\nThe best split is the split with the highest Gini reduction.")

ASSIGNMENT CONCLUSION
Parent Gini Impurity: 0.4688
Best Feature: Wind
Best Gini Reduction: 0.2812

The best split is the split with the highest Gini reduction.


In [25]:
import os

print("GENERATED FILES")

for file in [
    "tennis_gini_all_splits.csv",
    "tennis_gini_best_split_by_feature.csv"
]:
    if os.path.exists(file):
        print(file, "- Successfully Created")
    else:
        print(file, "- Not Found")

GENERATED FILES
tennis_gini_all_splits.csv - Successfully Created
tennis_gini_best_split_by_feature.csv - Successfully Created



# Gini Impurity Calculation for All Splits – Tennis Dataset

## Objective
To calculate Gini impurity for all possible splits in the tennis dataset and identify the best split using Gini reduction.

## Dataset Description
- Dataset: Unique Tennis Dataset
- Total Records: 16
- Target Column: Play
- Input Features: Outlook, Court, Wind, Temperature
- Problem Type: Binary Classification

## Gini Impurity Formula

Gini Impurity = 1 - Σ(pᵢ²)

## Calculations Performed

For every possible split, the following values were calculated:

1. Parent Gini Impurity – Measures the impurity of the complete dataset before splitting.
2. Left Child Gini Impurity – Measures the impurity of the left child node.
3. Right Child Gini Impurity – Measures the impurity of the right child node.
4. Weighted Child Gini Impurity – Calculates the weighted average impurity of both child nodes.

   Weighted Gini = (Left Weight × Left Gini) + (Right Weight × Right Gini)

5. Gini Reduction – Measures the decrease in impurity after splitting.

   Gini Reduction = Parent Gini - Weighted Child Gini

## Methodology

1. Loaded the tennis dataset using Pandas.
2. Identified the input features and target column.
3. Defined a function to calculate Gini impurity.
4. Calculated the parent Gini impurity.
5. Generated all unique possible binary splits.
6. Calculated left and right child Gini impurities.
7. Calculated weighted child Gini impurity.
8. Calculated Gini reduction for every split.
9. Identified the best split based on the highest Gini reduction.
10. Saved the results into CSV files.

## Results

- Calculated all five required Gini values for every generated split.
- Compared the Gini reduction values.
- Identified the best split based on maximum Gini reduction.
- Generated CSV files containing all split calculations and the best split for each feature.

## Conclusion

Gini impurity helps measure the quality of a split in a decision tree. A lower weighted child Gini indicates purer child nodes. The split with the highest Gini reduction is considered the best split because it provides the greatest reduction in impurity.

This assignment demonstrates how Gini impurity is used to select the best feature split when building a Decision Tree Classifier.
